# TPC-DS Benchmark Analysis
This notebook provides a visual comparison between Spark and Trino performance metrics based on the TPC-DS benchmark results.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import json

# Set paths
BASE_DIR = Path("../benchmark/results/raw")
SPARK_RESULTS = BASE_DIR / "spark_results.jsonl"
TRINO_RESULTS = BASE_DIR / "trino_results.jsonl"

def load_results(path):
    if not path.exists():
        return pd.DataFrame()
    with path.open("r") as f:
        data = [json.loads(line) for line in f if line.strip()]
    return pd.DataFrame(data)

spark_df = load_results(SPARK_RESULTS)
trino_df = load_results(TRINO_RESULTS)

df = pd.concat([spark_df, trino_df], ignore_index=True)
df = df[df['run_type'] == 'measured']
df['peak_memory_mb'] = df['peak_memory_bytes'] / (1024 * 1024)

print(f"Loaded {len(df)} measured records.")
df.head()

Loaded 198 measured records.


,engine,query_name,run_type,run_number,query_id,status,query_time_seconds,throughput_qps,peak_memory_bytes,spill_bytes,cpu_time_millis,result_hash,row_count,error_message,peak_memory_mb
0,spark,query1,measured,1,benchmark-query1-run-1,success,7.250,0.137931,35913696,0,2896,200428bfb5199070f46b63451dc31e5b,100,,34.249969
1,spark,query2,measured,1,benchmark-query2-run-1,success,8.275,0.120846,17078152,0,3143,1a57aeeeff7bf0872f86d811adf24478,2513,,16.286995
2,spark,query3,measured,1,benchmark-query3-run-1,success,6.408,0.156055,17039344,0,2207,6c4e79b7e727e5f1e9d340012399f6e8,61,,16.249985
3,spark,query4,measured,1,benchmark-query4-run-1,success,19.654,0.050880,106954656,0,14370,327e17dc996692fb72fe337a73fb881f,8,,101.999908
4,spark,query5,measured,1,benchmark-query5-run-1,success,12.318,0.081182,34078688,0,4576,5367df38a05b44a27cbe8c0c60cbfb7f,100,,32.499969


## 1. Execution Time Comparison (Latency)
Lower is better.

In [2]:
avg_latencies = df.groupby(['query_name', 'engine'])['query_time_seconds'].mean().reset_index()

fig = px.bar(avg_latencies, x='query_name', y='query_time_seconds', color='engine', 
             barmode='group', title='Average Query Execution Time (Seconds)',
             labels={'query_time_seconds': 'Seconds', 'query_name': 'Query ID'},
             template='plotly_dark')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 2. Memory Consumption
Comparing peak execution memory (MB).

In [3]:
fig_mem = px.box(df, x='engine', y='peak_memory_mb', color='engine', 
                 points='all', title='Peak Memory Distribution (MB)',
                 template='plotly_dark')
fig_mem.show()

## 3. Performance Ratio (Spark/Trino)
How many times faster is Trino than Spark?

In [4]:
pivot_time = avg_latencies.pivot(index='query_name', columns='engine', values='query_time_seconds')
pivot_time['ratio'] = pivot_time['spark'] / pivot_time['trino']
pivot_time = pivot_time.dropna().reset_index()

fig_ratio = px.line(pivot_time, x='query_name', y='ratio', 
                    title='Performance Ratio (Spark Time / Trino Time)',
                    labels={'ratio': 'Ratio (S/T)', 'query_name': 'Query ID'},
                    markers=True, template='plotly_dark')
fig_ratio.add_hline(y=1, line_dash="dash", line_color="red", annotation_text="Parity")
fig_ratio.show()

print(f"Average Ratio: {pivot_time['ratio'].mean():.2f}x")

Average Ratio: 7.95x
